**14장 – 합성곱 신경망을 사용한 컴퓨터 비전**

# 설정

먼저 몇 개의 모듈을 임포트합니다. 맷플롯립 그래프를 인라인으로 출력하도록 만들고 그림을 저장하는 함수를 준비합니다. 또한 파이썬 버전이 3.5 이상인지 확인합니다(파이썬 2.x에서도 동작하지만 곧 지원이 중단되므로 파이썬 3을 사용하는 것이 좋습니다). 사이킷런 버전이 0.20 이상인지와 텐서플로 버전이 2.0 이상인지 확인합니다.

In [ ]:
# 파이썬 ≥3.5 필수
import sys
assert sys.version_info >= (3, 5)

# 사이킷런 ≥0.20 필수
import sklearn
assert sklearn.__version__ >= "0.20"

# 코랩에서 실행되는 노트북인가요?
IS_COLAB = "google.colab" in sys.modules

# 텐서플로 ≥2.0 필수
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

if not tf.config.list_physical_devices('GPU'):
    print("감지된 GPU가 없습니다. GPU가 없으면 CNN은 매우 느릴 수 있습니다.")
    if IS_COLAB:
        print("런타임 > 런타임 유형 변경 메뉴를 선택하고 하드웨어 가속기로 GPU를 고르세요.")

# 공통 모듈 임포트
import numpy as np
import os

# 노트북 실행 결과를 동일하게 유지하기 위해
np.random.seed(42)
tf.random.set_seed(42)

# 깔끔한 그래프 출력을 위해
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# 그림을 저장할 위치
PROJECT_ROOT_DIR = "."
CHAPTER_ID = "cnn"
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images", CHAPTER_ID)
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("그림 저장", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

흑백 이미지와 컬러 이미지 출력을 위한 유틸리티 함수:

In [ ]:
def plot_image(image):
    plt.imshow(image, cmap="gray", interpolation="nearest")
    plt.axis("off")

def plot_color_image(image):
    plt.imshow(image, interpolation="nearest")
    plt.axis("off")

# 합성곱이란 무엇인가?

In [ ]:
import numpy as np
from sklearn.datasets import load_sample_image

# 샘플 이미지를 로드합니다.
china = load_sample_image("china.jpg") / 255
flower = load_sample_image("flower.jpg") / 255

plt.figure(figsize=(20,20))
plt.subplot(1,2,1)      # plt.subplot(nrows, ncols, index)
plot_color_image(china)
plt.subplot(1,2,2)
plot_color_image(flower)

In [ ]:
images = np.array([china, flower])
batch_size, height, width, channels = images.shape
# images  : [batch, in_height, in_width, in_channels]

In [ ]:
images[0].shape

In [ ]:
china.shape

In [ ]:
flower.shape

In [ ]:
plt.figure(figsize=(15,10))
plt.subplot(2,2,1)
plot_image(images[1])
plt.subplot(2,2,2)
plot_image(images[1, :, :, 0])
plt.subplot(2,2,3)
plot_image(images[1, :, :, 1])
plt.subplot(2,2,4)
plot_image(images[1, :, :, 2])

In [ ]:
# 2개의 필터를 만듭니다.
filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)
filters[:, 3, :, 0] = 1  # 수직선
filters[3, :, :, 1] = 1  # 수평선
# filters : [filter_height, filter_width, in_channels, out_channels]

outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

plt.figure(figsize=(20,20))
plt.subplot(1,2,1)
plot_image(outputs[1, :, :, 0])
plt.subplot(1,2,2)
plot_image(outputs[1, :, :, 1])

[tf.nn.conv2d](https://runebook.dev/en/docs/tensorflow/compat/v1/nn/conv2d)

tf.nn.conv2d(images, filters, strides, padding)
* images : [batch, height, width, channel]
* filters : [height, width, inp_channel, out_channels]
* strides : 1 or 4개의 원소를 가진 1D 배열. [ batch stride, height stride, width stride, channel stride ]


In [ ]:
# 2개의 필터를 만듭니다.
filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)
filters[:, 3, :, 0] = 1  # 수직선
filters[3, :, :, 1] = 1  # 수평선
# filters : [filter_height, filter_width, in_channels, out_channels]

In [ ]:
filters

In [ ]:
filters[0]

In [ ]:
filters[0][0]

In [ ]:
filters[0].shape

<img src="https://drive.google.com/uc?export=view&id=13wLm5gUog4uWTsFhpp5UdeLtwg783XbZ" width=600/>
*7 x 7 x 3 x 2 filters*

정리해보면, 3개의 채널을 가진 7 x 7 필터를 2개 만든것입니다.

In [ ]:
print(filters[0].shape)

In [ ]:
def inspectFilters(filters, f=None):
    if f:
        if filters.shape[3] <= f:
            print('no filters for that index')
            return 1
    print("[")
    for x in filters:               # 7 x 7 x 3 x 2에서 첫 7에 대해 하나의 열씩
        print(" [")
        n_y = x.shape[0]            # 7 x 3 x 2에서 7에 해당
        n_ch = x.shape[1]           # 7 x 3 x 2에서 3에 해당
        for c in range(n_ch):       # 7 x 3 x 2에서 3에 대해, 즉 RGB 하나씩에 대해
            for i in range(n_y):
                print("  [" if c==0 else "   ", end='')
                print(x[i, c, f] if f or f == 0 else x[i, c] , end='')
                print('] 'if c == n_ch - 1 else '  ', end='')
            print('')
        print(" ]")
    print("]")

In [ ]:
inspectFilters(filters)

In [ ]:
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

plt.figure(figsize=(20,20))
plt.subplot(1,2,1)
plot_image(outputs[0, :, :, 0])
plt.subplot(1,2,2)
plot_image(outputs[0, :, :, 1])

In [ ]:
# edge 필터를 만듭니다.
filters = np.zeros(shape=(7, 7, channels, 1), dtype=np.float32)
filters[0, :, :, 0] = 1  # 수평선
filters[6, :, :, 0] = -1  # 수평선
# filters : [filter_height, filter_width, in_channels, out_channels]
inspectFilters(filters)

In [ ]:
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

plt.figure(figsize=(10,10))
plot_image(outputs[0, :, :, 0])

In [ ]:
# 2개의  필터를 만듭니다.
filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)
filters[0, :, :, 0] = -1
filters[0, 3, :, 0] = -2
filters[6, :, :, 0] = 1
filters[6, 3, :, 0] = 2
filters[:, 0, :, 1] = -1
filters[:, 6, :, 1] = 1
filters[3, 0, :, 1] = -2
filters[3, 6, :, 1] = 2
inspectFilters(filters)

In [ ]:
inspectFilters(filters)

In [ ]:
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

plt.figure(figsize=(20,20))
plt.subplot(1,2,1)
plot_image(outputs[0, :, :, 0])
plt.subplot(1,2,2)
plot_image(outputs[0, :, :, 1])

In [ ]:
filters = np.zeros(shape=(7, 7, channels, 2), dtype=np.float32)
filters[:, 3, :, 0] = 1  # 수직선
filters[3, :, :, 1] = 1  # 수평선
# filters : [filter_height, filter_width, in_channels, out_channels]
inspectFilters(filters,0)

In [ ]:
outputs = tf.nn.conv2d(images, filters, strides=1, padding="SAME")

plt.figure(figsize=(20,15))
for image_index in (0, 1):
    for feature_map_index in (0, 1):
        plt.subplot(2, 2, image_index * 2 + feature_map_index + 1)
        plot_image(outputs[image_index, :, :, feature_map_index])

plt.show()

In [ ]:
def crop(images):
    return images[150:220, 130:250]

In [ ]:
plot_image(crop(images[0, :, :, 0]))
save_fig("china_original", tight_layout=False)
plt.show()

for feature_map_index, filename in enumerate(["china_vertical", "china_horizontal"]):
    plot_image(crop(outputs[0, :, :, feature_map_index]))
    save_fig(filename, tight_layout=False)
    plt.show()

In [ ]:
plot_image(filters[:, :, 0, 0])
plt.show()
plot_image(filters[:, :, 0, 1])
plt.show()

In [ ]:
inspectFilters(filters, 0)

## 합성곱 층

`keras.layers.Conv2D()`를 사용해 2D 합성곱 층을 만들어 봅시다.

In [ ]:
np.random.seed(30)
tf.random.set_seed(30)

conv = keras.layers.Conv2D(filters=2, kernel_size=7, strides=1,
                           padding="SAME", activation="relu", input_shape=outputs.shape)

두 개의 테스트 이미지로 이 층을 호출합니다:

In [ ]:
conv_outputs = conv(images)
conv_outputs.shape

출력은 4D 텐서입니다. 차원은 배치 크기, 높이, 너비, 채널입니다. 2개의 이미지를 입력으로 사용했기 때문에 첫 번째 차원(배치 크기)는 2입니다. 다음 두 차원은 출력 특성맵의 높이와 너비입니다. `padding="SAME"`와 `strides=1`로 설정했기 때문에 출력 특성맵의 높이와 너비는 입력 이미지와 같습니다(이 경우 427×640). 마지막으로 이 합성곱 층은 2개의 필터를 사용합니다. 따라서 마지막 차원의 크기는 2입니다. 즉 입력 이미지마다 2개의 특성맵이 출력됩니다.

필터는 초기에 랜덤하게 초기화되기 때문에 처음에는 랜덤한 패턴을 감지합니다. 이미지마다 출력된 2개의 특성맵을 확이해 봅시다:

In [ ]:
plt.figure(figsize=(10,6))
for image_index in (0, 1):
    for feature_map_index in (0, 1):
        plt.subplot(2, 2, image_index * 2 + feature_map_index + 1)
        plot_image(crop(conv_outputs[image_index, :, :, feature_map_index]))
plt.show()

필터가 초기에 랜덤하게 초기화되엇지만 어떤 경우에의 필터는 에지를 감지한 것처럼 보입니다. 랜덤하게 초기화된 필터는 종종 이런 식으로 동작합니다. 에지 감지는 이미지 처리에 매우 유용하기 때문에 운이 좋습니다.

원한다면 필터를 앞에서 수동으로 정의한 필터를 사용하고 편향을 0으로 지정할 수 있습니다(합성곱 층이 훈련하는 동안 적절한 필터와 편향을 학습하기 때문에 실제로는 수동으로 필터와 편향을 지정할 필요가 거의 없습니다):

이제 이 층을 동일한 두 이미지에서 다시 호출해 봅시다. 그다음 출력 특성맵이 (앞에서처럼) 수직선과 수평선을 부각하는지 확인해 보겠습니다:

In [ ]:
conv_outputs = conv(images)
conv_outputs.shape

In [ ]:
plt.figure(figsize=(10,6))
for image_index in (0, 1):
    for feature_map_index in (0, 1):
        plt.subplot(2, 2, image_index * 2 + feature_map_index + 1)
        plot_image(crop(conv_outputs[image_index, :, :, feature_map_index]))
plt.show()

## VALID vs SAME 패딩

In [ ]:
def feature_map_size(input_size, kernel_size, strides=1, padding="SAME"):
    if padding == "SAME":
        return (input_size - 1) // strides + 1
    else:
        return (input_size - kernel_size) // strides + 1

In [ ]:
def pad_before_and_padded_size(input_size, kernel_size, strides=1):
    fmap_size = feature_map_size(input_size, kernel_size, strides)
    padded_size = max((fmap_size - 1) * strides + kernel_size, input_size)
    pad_before = (padded_size - input_size) // 2
    return pad_before, padded_size

In [ ]:
def manual_same_padding(images, kernel_size, strides=1):
    if kernel_size == 1:
        return images.astype(np.float32)
    batch_size, height, width, channels = images.shape
    top_pad, padded_height = pad_before_and_padded_size(height, kernel_size, strides)
    left_pad, padded_width  = pad_before_and_padded_size(width, kernel_size, strides)
    padded_shape = [batch_size, padded_height, padded_width, channels]
    padded_images = np.zeros(padded_shape, dtype=np.float32)
    padded_images[:, top_pad:height+top_pad, left_pad:width+left_pad, :] = images
    return padded_images

`"SAME"` 패딩을 사용하는 것은 `manual_same_padding()`을 사용해 수동으로 패딩하고 `"VALID"` 패딩을 사용하는 것과 동일합니다(혼동될 수 있지만 `"VALID"` 패딩은 전혀 패딩을 하지 않는다는 뜻입니다):

In [ ]:
kernel_size = 7
strides = 2

conv_valid = keras.layers.Conv2D(filters=1, kernel_size=kernel_size, strides=strides, padding="VALID")
conv_same = keras.layers.Conv2D(filters=1, kernel_size=kernel_size, strides=strides, padding="SAME")

valid_output = conv_valid(manual_same_padding(images, kernel_size, strides))

# conv_same의 가중치를 생성하기 위해 build() 메서드를 호출해야 합니다.
conv_same.build(tf.TensorShape(images.shape))

# conv_valid의 가중치를 conv_same으로 복사합니다.
conv_same.set_weights(conv_valid.get_weights())

same_output = conv_same(images.astype(np.float32))

assert np.allclose(valid_output.numpy(), same_output.numpy())

In [ ]:
valid_output.shape

stride를 변경하며 shape을 다시 살펴보시오.


# 풀링 층

### 최대 풀링

In [ ]:
max_pool = keras.layers.MaxPool2D(pool_size=(2,2))
max_pool.pool_size

In [ ]:
# 혹은 간단히 이렇게 사용할수도 있다
max_pool = keras.layers.MaxPool2D(pool_size=2)
max_pool.pool_size

In [ ]:
cropped_images = np.array([crop(image) for image in images], dtype=np.float32)
output = max_pool(cropped_images)

In [ ]:
fig = plt.figure(figsize=(12, 8))
gs = mpl.gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("Input", fontsize=14)
ax1.imshow(cropped_images[0])  # 첫 번째 이미지 그리기
ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("Output", fontsize=14)
ax2.imshow(output[0])  # 첫 번째 이미지 출력 그리기
ax2.axis("off")
save_fig("china_max_pooling")
plt.show()

### 깊이 방향(depth-wise) 풀링


최대풀링과 평균풀링등을 공간차원 즉 주어진 매칭 되는 특성맵이 아닌 깊이 차원 즉 하위 합성곱층내 여러개의 특성맵들을 대해 적용을 할수 있는데, 이를 깊이 방향 풀링이라 한다.


![DepthPooling](https://drive.google.com/uc?export=view&id=1P53SeDeFTzSKDJ7NYhPo5714leiB_tQa)DepthPooling

깊이 방향 풀링은 동일 패턴이 회전된 여러가지 필터를 학습 했을때, 회전에 상관없이 동일 출력을 낸다

회전, 두께, 밝기, 왜곡, 색상 등에 대한 불변성을 학습할수 있다.


In [ ]:
class DepthMaxPool(keras.layers.Layer):
    def __init__(self, pool_size, strides=None, padding="VALID", **kwargs):
        super().__init__(**kwargs)
        if strides is None:
            strides = pool_size
        self.pool_size = pool_size
        self.strides = strides
        self.padding = padding
    def call(self, inputs):
        return tf.nn.max_pool(inputs,
                              ksize=(1, 1, 1, self.pool_size),
                              strides=(1, 1, 1, self.pool_size),
                              padding=self.padding)

커널과 스트라이드가 (1, 1, 1, x) 즉 앞의 배치, 높이, 너비 차원인것을 말한다.

In [ ]:
depth_pool = DepthMaxPool(3)
with tf.device("/cpu:0"): # 아직 GPU 커널이 없습니다.
    depth_output = depth_pool(cropped_images)
depth_output.shape

In [ ]:
plt.figure(figsize=(12, 8))
plt.subplot(1, 2, 1)
plt.title("Input", fontsize=14)
plot_color_image(cropped_images[0])  # 첫 번째 이미지 그리기
plt.subplot(1, 2, 2)
plt.title("Output", fontsize=14)
plot_color_image(depth_output[0, ..., 0])  # 첫 번째 이미지 출력 그리기
plt.axis("off")
plt.show()

또는 `Lambda` 층을 사용합니다:

깊이차원 풀링 만들기
<pre>
tf.nn.max_pool(
    input, ksize, strides, padding, data_format=None, name=None
)
</pre>
* ksize : [batch, height, width, depth_stride]
* strides : [batch, height, width, depth_stride]

In [ ]:
depth_pool = keras.layers.Lambda(lambda X: tf.nn.max_pool(
    X, ksize=(1, 1, 1, 3), strides=(1, 1, 1, 3), padding="VALID"))
with tf.device("/cpu:0"): # 아직 GPU 커널이 없습니다.
    depth_output = depth_pool(cropped_images)
depth_output.shape

In [ ]:
fig = plt.figure(figsize=(12, 8))
gs = mpl.gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("Input", fontsize=14)
ax1.imshow(cropped_images[0])  # 첫 번째 이미지 그리기
ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("Output", fontsize=14)
ax2.imshow(depth_output[0, ..., 0])  # 첫 번째 이미지 출력 그리기
ax2.axis("off")
plt.show()

### 평균 풀링

In [ ]:
avg_pool = keras.layers.AvgPool2D(pool_size=2)
avg_pool.pool_size

In [ ]:
output_avg = avg_pool(cropped_images)

In [ ]:
fig = plt.figure(figsize=(12, 8))
gs = mpl.gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[2, 1])

ax1 = fig.add_subplot(gs[0, 0])
ax1.set_title("Input", fontsize=14)
ax1.imshow(cropped_images[0])  # 첫 번째 이미지 그리기
ax1.axis("off")
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_title("Output", fontsize=14)
ax2.imshow(output_avg[0])  # 첫 번째 이미지 출력 그리기
ax2.axis("off")
plt.show()

In [ ]:
max_pool = keras.layers.MaxPool2D(pool_size=2)
output_max = avg_pool(cropped_images)

plt.figure(figsize=(20, 15))
plt.subplot(1, 2, 1)
plt.title("Avg", fontsize=14)
plot_color_image(output_avg[0])  # 첫 번째 이미지 그리기
plt.subplot(1, 2, 2)
plt.title("Max", fontsize=14)
plot_color_image(output_max[0])  # 첫 번째 이미지 출력 그리기
plt.axis("off")
plt.show()

### 전역 평균 풀링

In [ ]:
global_avg_pool = keras.layers.GlobalAvgPool2D()
global_avg_pool(cropped_images)

In [ ]:
output_global_avg2 = keras.layers.Lambda(lambda X: tf.reduce_mean(X, axis=[1, 2]))
output_global_avg2(cropped_images)

# CNN으로 패션 MNIST 문제 풀기

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

plt.subplot(1,2,1)
plt.imshow(X_train[0], cmap="binary")

X_mean = X_train.mean(axis=0, keepdims=True)
X_std = X_train.std(axis=0, keepdims=True) + 1e-7
X_train = (X_train - X_mean) / X_std
X_valid = (X_valid - X_mean) / X_std
X_test = (X_test - X_mean) / X_std

X_train = X_train[..., np.newaxis]
X_valid = X_valid[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train[0][:,:,0].shape)
plt.subplot(1,2,2)
plt.imshow(X_train[0][:,:,0], cmap="binary")


In [ ]:
a=np.array([[2,3],[4,5]])
c=np.array([[[2,3],[4,5]],[[6,7],[8,9]]])
b=c[..., np.newaxis]
b

In [ ]:
from tensorflow.keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Dropout, Flatten
from keras.layers import MaxPooling2D, Conv2D

model = Sequential()
model.add(Conv2D(32, kernel_size=(3,3), input_shape=[28, 28, 1], padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(64, kernel_size=(3,3), padding='same'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Flatten())
model.add(Dense(128))
model.add(Activation('relu'))

model.add(Dropout(0.2))
model.add(Dense(10))
model.add(Activation('softmax'))

model.summary()

Conv Layer Parameters=(Kernel Width×Kernel Height×Input Channels+1)×Output Channels

why the first conv2d has 320 parameters?
* (3×3×1+1)×32=(9+1)×32=10×32=320

why the second conv2d has 18,496 parameters?
* (3×3×32+1)×64 = (9×32+1)×64 = (288+1)×64 = 289×64 = 18,496

why flatten layer has 3,136 neurons?
* 7 x 7 x 64 = 3136

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))
score = model.evaluate(X_test, y_test)
X_new = X_test[:10] # 새로운 이미지처럼 사용합니다
y_pred = model.predict(X_new)

In [ ]:
y_pred.shape

In [ ]:
y_pred[0]

In [ ]:
X_new.shape

In [ ]:
i = 0   # i 에 확인하고 싶은 인덱스를 넣어서 예측값 y_pred[i]와 실제값 X_new[i]을 비교 해보자
print(y_pred[i])
ind = np.argmax(y_pred[i])
print(ind)
print(class_names[ind])
plot_image(X_new[i, :, :, 0])

### 일반 FCN

In [ ]:
fcn = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(200, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])
fcn.summary()